# Entrenamiento HMM - Weather para varios K
Entrena Baum-Welch con K = {5, 8, 10, 12} sobre datos RevIN-normalizados (per-window) y guarda cache para cada K.

In [1]:
import numpy as np
import pandas as pd
import torch
import sys
import time
sys.path.insert(0, '..')
from hmm.baum_welch import baum_welch

In [2]:
# Cargar train set (70% split como Dataset_Custom) y normalizar con RevIN per-window
df = pd.read_csv('../dataset/weather/weather.csv')
n_train = int(len(df) * 0.7)
train_ot = df['OT'].values[:n_train]

seq_len = 96
windows = []
for start in range(0, len(train_ot) - seq_len + 1, seq_len):
    window = train_ot[start:start + seq_len]
    w_mean, w_std = window.mean(), max(window.std(), 1e-5)
    windows.append((window - w_mean) / w_std)

train_norm = np.concatenate(windows)
print(f'Train: {len(train_norm)} timesteps ({len(windows)} ventanas x {seq_len})')
print(f'RevIN-normalized: mean={train_norm.mean():.6f}, std={train_norm.std():.6f}')

Train: 36864 timesteps (384 ventanas x 96)
RevIN-normalized: mean=-0.000000, std=1.000000


In [3]:
# Entrenar y cachear para cada K (skip si ya existe en cache)
import os
K_values = [5, 8, 10, 12]

for K in K_values:
    path = f'../cache/hmm_weather_K{K}.pth'

    if os.path.exists(path):
        c = torch.load(path, weights_only=False)
        if c['converged']:
            print(f'K={K}: CACHE EXISTENTE (convergido) - iter {c["n_iter"]}, LL={c["log_likelihood"]:.2f}')
            continue
        else:
            print(f'K={K}: cache existe pero NO convergio - reentrenando...')

    print(f'\n{"="*60}')
    print(f'Entrenando K={K}')
    print(f'{"="*60}')

    t0 = time.time()
    result = baum_welch(
        observations=train_norm,
        K=K,
        max_iter=2000,
        epsilon=1e-4,
        random_state=42,
        verbose=True
    )
    elapsed = time.time() - t0

    # Validar
    assert result['converged'], f'K={K} NO CONVERGIO en {result["n_iter"]} iters'
    assert result['log_likelihood'] != float('-inf'), f'K={K} log_likelihood es -inf'
    assert np.all(result['sigma'] > 0), f'K={K} sigma tiene valores <= 0'
    assert np.allclose(result['A'].sum(axis=1), 1.0, atol=1e-6), f'K={K} filas de A no suman 1'

    # Guardar cache
    cache = {
        'A': torch.from_numpy(result['A']).float(),
        'pi': torch.from_numpy(result['pi']).float(),
        'mu': torch.from_numpy(result['mu']).float(),
        'sigma': torch.from_numpy(result['sigma']).float(),
        'log_likelihood': result['log_likelihood'],
        'converged': result['converged'],
        'n_iter': result['n_iter'],
    }
    torch.save(cache, path)

    print(f'\nK={K}: converged iter {result["n_iter"]}, LL={result["log_likelihood"]:.2f}, tiempo={elapsed:.1f}s')
    print(f'  mu: {result["mu"]}')
    print(f'  sigma: {result["sigma"]}')
    print(f'  Cache guardado: {path}')

K=5: CACHE EXISTENTE (convergido) - iter 159, LL=-10762.65
K=8: CACHE EXISTENTE (convergido) - iter 329, LL=-1286.77

Entrenando K=10


Baum-Welch EM:  21%|██        | 423/2000 [4:53:29<18:14:10, 41.63s/it, LL=2982.69, ΔLL=1.06e-04]



Convergió en iteración 424/2000
  Log-likelihood final: 2982.6887

K=10: converged iter 424, LL=2982.69, tiempo=17610.6s
  mu: [-0.40355091 -0.88660153  0.38221891  1.31265818 -0.66992609 -1.51390828
  0.83043825 -1.13716335 -0.03624659  2.05807129]
  sigma: [0.10742698 0.07215043 0.15136349 0.19426666 0.0759929  0.25579883
 0.15916328 0.09156029 0.13686467 0.51918401]
  Cache guardado: ../cache/hmm_weather_K10.pth

Entrenando K=12


Baum-Welch EM:  32%|███▏      | 632/2000 [8:34:24<18:33:28, 48.84s/it, LL=5301.61, ΔLL=1.04e-04]


Convergió en iteración 633/2000
  Log-likelihood final: 5301.6056

K=12: converged iter 633, LL=5301.61, tiempo=30866.0s
  mu: [ 1.40898705 -0.54846367  0.36327483 -1.52188965 -1.15050427  0.01812118
 -0.72775163 -0.30083041  1.02210487  2.09559496  0.70117855 -0.9111743 ]
  sigma: [0.1698682  0.06928111 0.12298617 0.25647324 0.08893917 0.11785947
 0.06055197 0.10071961 0.12519491 0.52518673 0.11908418 0.06810939]
  Cache guardado: ../cache/hmm_weather_K12.pth


In [4]:
# Resumen comparativo
print(f'{"K":>3} | {"Iters":>5} | {"LL":>12} | {"sigma_min":>10} | {"sigma_max":>10} | {"A_diag_mean":>11}')
print('-' * 70)
for K in K_values:
    path = f'../cache/hmm_weather_K{K}.pth'
    if not os.path.exists(path):
        print(f'{K:3d} | pendiente')
        continue
    c = torch.load(path, weights_only=False)
    sigma = c['sigma'].numpy()
    A_diag = torch.diag(c['A']).numpy()
    print(f'{K:3d} | {c["n_iter"]:5d} | {c["log_likelihood"]:12.2f} | {sigma.min():10.4f} | {sigma.max():10.4f} | {A_diag.mean():11.4f}')

  K | Iters |           LL |  sigma_min |  sigma_max | A_diag_mean
----------------------------------------------------------------------
  5 |   159 |    -10762.65 |     0.1421 |     0.5154 |      0.9147
  8 |   329 |     -1286.77 |     0.0900 |     0.5159 |      0.8556
 10 |   424 |      2982.69 |     0.0722 |     0.5192 |      0.8225
 12 |   633 |      5301.61 |     0.0606 |     0.5252 |      0.7708
